In [ ]:
# S1 — Level-specific wrapping fractions across the kappa sweep
# 
# At resonance (kf=1.0), the wrapping at ci=11 (gen2) is:
#   R0: 0/2 = 0%,  R1: ~1.5/3 = 50%,  R2: ~3.67/5 = 73.3%,  R3: 6/7 = 85.7%
# These fractions encode the primes through the number of sheets p_{k+1}.
# How do they change with kappa, and what happens at the resonance?

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa_natural = 1.0 / np.sqrt(P4)
omega = 2 * np.pi

cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

from solenoid_jax import warmup
warmup()

# The wrapping fraction at level k, crossing ci depends on:
# - Number of sheets: p_{k+1}
# - Comb spacing: delta = 2*pi * e^{-kappa*ci}
# - Base value at that level
# 
# For the analytic COMB MODEL:
# Values = base + j * delta, j=0,...,n_sheets-1
# Sheet j wraps if |base + j*delta| > pi (modulo 2pi)
#
# At ci_g2 = 11 with natural kappa:
# delta = 2*pi * e^{-11/sqrt(210)} = 2*pi * 0.468 = 2.941
# This is LESS than pi. So not all sheets wrap.
# The first non-wrapping sheet is j=0 (base ≈ 0 at R3).
# Sheet j wraps if j * delta > pi - base, i.e., j > (pi - base) / delta

delta_natural = 2 * np.pi * np.exp(-kappa_natural * 11)
print(f'Comb spacing at ci=11, natural kappa: delta = {delta_natural:.4f}')\n",
    "print(f'pi = {np.pi:.4f}')\n",
    "print(f'delta/pi = {delta_natural/np.pi:.4f}')\n",
    "print(f'\\nSheets that wrap at each level (for small positive base):')\n",
    "for k, p_next in enumerate([p1, p2, p3, p4]):\n",
    "    n_sheets = p_next\n",
    "    # Sheets 0,...,n_sheets-1. Sheet j wraps if j*delta > pi (approx)\n",
    "    j_threshold = np.pi / delta_natural\n",
    "    n_wrap = sum(1 for j in range(n_sheets) if j * delta_natural > np.pi * 0.95)  # rough\n",
    "    frac = n_wrap / n_sheets\n",
    "    print(f'  R{k} ({n_sheets} sheets): j_threshold={j_threshold:.2f}, ~{n_wrap}/{n_sheets} wrap = {frac*100:.0f}%')\n",
    "\n",
    "# ══════════════════════════════════════════════════════════════\n",
    "# KEY INSIGHT: the wrapping threshold j > pi/delta determines how\n",
    "# many sheets wrap. At resonance, delta = 2.94 ≈ pi, so:\n",
    "# j_threshold = pi/delta ≈ 1.07\n",
    "# This means sheets j >= 2 always wrap, and j=1 is right at the boundary.\n",
    "#\n",
    "# The number of wrapping sheets at level k = p_{k+1} - ceil(j_threshold)\n",
    "# = p_{k+1} - 2 (for delta near pi, j_threshold ≈ 1)\n",
    "# \n",
    "# Non-wrapping sheets: j=0 always, j=1 partially.\n",
    "# Wrapping fraction = (p_{k+1} - 1 - partial) / p_{k+1}\n",
    "# At R3: (7 - 1) / 7 = 6/7 = 85.7% ✓\n",
    "# At R2: (5 - 1 - partial) / 5 ≈ 3.67/5 = 73.3%\n",
    "# At R1: (3 - 1 - partial) / 3 ≈ 1.5/3 = 50%\n",
    "# At R0: (2 - 1 - partial) / 2 ≈ 0/2 = 0% (both sheets below pi)\n",
    "#\n",
    "# The \"partial\" wrapping of j=1 is what creates the FINE STRUCTURE.\n",
    "# ══════════════════════════════════════════════════════════════\n",
    "\n",
    "# Now: what's the ANALYTIC wrapping fraction as a function of kappa?\n",
    "# At level k with p_{k+1} sheets:\n",
    "# Wrapping fraction f_k(kappa) = (p_{k+1} - 1) / p_{k+1}  when delta >> pi (all but j=0 wrap)\n",
    "#                               = 0  when delta << pi/p_{k+1} (no wrapping)\n",
    "# The transition happens when delta ~ pi, i.e., e^{-kappa*ci} ~ 1/2\n",
    "\n",
    "# Let me compute the ACTUAL wrapping fraction at ci=11 for each level\n",
    "# across the kappa sweep, using the full cascade (not the comb model).\n",
    "\n",
    "print(f'\\n=== Wrapping fraction at ci=11 (gen2) vs kappa ===')\n",
    "kf_values = [0.8, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2]\n",
    "ci_g2_idx = np.where(cis == 11)[0][0]\n",
    "\n",
    "print(f'{\"kf\":>5} {\"delta\":>8} {\"d/pi\":>6} {\"fR0\":>6} {\"fR1\":>6} {\"fR2\":>6} {\"fR3\":>6}  {\"x_q(R0)\":>10}')\n",
    "for kf in kf_values:\n",
    "    kv = kappa_natural * kf\n",
    "    delta = 2 * np.pi * np.exp(-kv * 11)\n",
    "    \n",
    "    sys0 = SolenoidSystem(primes=primes, kappa=kv, epsilon=kv)\n",
    "    branches = sys0.all_branches()\n",
    "    res = sys0.integrate_all_branches(branches, cis.astype(float), float(P4+1), backend='jax')\n",
    "    \n",
    "    # Wrapping fractions at ci=11\n",
    "    fracs = []\n",
    "    for k in range(4):\n",
    "        Rk = np.array([res[br][ci_g2_idx, k] for br in branches])\n",
    "        f = np.sum(np.abs(Rk) > np.pi) / len(branches)\n",
    "        fracs.append(f)\n",
    "    \n",
    "    # x_q(R0)\n",
    "    rms_all = np.zeros((len(cis), 4))\n",
    "    for idx in range(len(cis)):\n",
    "        for k in range(4):\n",
    "            Rk = np.array([res[br][idx, k] for br in branches])\n",
    "            Rk_w = np.mod(Rk, 2*np.pi)\n",
    "            Rk_w[Rk_w > np.pi] -= 2*np.pi\n",
    "            rms_all[idx, k] = np.sqrt(np.mean(Rk_w**2))\n",
    "    \n",
    "    i1 = np.where((a3==1)&(a5==0)&(a7==4))[0][0]  # gen2\n",
    "    i2 = np.where((a3==1)&(a5==0)&(a7==2))[0][0]  # gen3\n",
    "    cp_R0 = rms_all[i1, 0] / rms_all[i2, 0]\n",
    "    x_R0 = np.log(20.0) / np.log(cp_R0) if cp_R0 > 1.01 else np.nan\n",
    "    \n",
    "    print(f'{kf:5.2f} {delta:8.4f} {delta/np.pi:6.3f} {fracs[0]*100:5.1f}% {fracs[1]*100:5.1f}% '\n",
    "          f'{fracs[2]*100:5.1f}% {fracs[3]*100:5.1f}%  {x_R0:10.6f}')\n",
    "\n",
    "# ══════════════════════════════════════════════════════════════\n",
    "# The wrapping fraction at each level follows the formula:\n",
    "# f_k ≈ (p_{k+1} - n_unwrap) / p_{k+1}\n",
    "# where n_unwrap depends on delta/pi.\n",
    "#\n",
    "# At resonance (delta/pi ≈ 0.936):\n",
    "# - R0 (2 sheets): both below pi → f=0\n",
    "# - R1 (3 sheets): 1 fully + 0.5 partially → f=50%\n",
    "# - R2 (5 sheets): 3 fully + 0.67 partially → f=73%\n",
    "# - R3 (7 sheets): 5 fully + 1.0 at boundary → f=86%\n",
    "#\n",
    "# The wrapping fractions encode the primes:\n",
    "# f_k = (p_{k+1} - c_k) / p_{k+1}\n",
    "# where c_k is the effective number of non-wrapping sheets.\n",
    "# ══════════════════════════════════════════════════════════════\n",
    "\n",
    "print(f'\\n=== At resonance (kf=1.0): wrapping = prime structure ===')\n",
    "print(f'delta/pi = {delta_natural/np.pi:.4f}')\n",
    "for k, p_next in enumerate([p1, p2, p3, p4]):\n",
    "    if k == 0:\n",
    "        f_theory = 0.0  # 2 sheets, both below pi\n",
    "    else:\n",
    "        # Number of sheets with j*delta > pi: those with j > pi/delta = 1.069\n",
    "        # So j >= 2 wraps, j=1 is partially wrapping\n",
    "        # For p sheets: full wrappers = p - 2, partial = 1\n",
    "        # f ≈ (p-2 + frac)/p where frac = portion of j=1 that wraps\n",
    "        pass\n",
    "    print(f'  R{k}: {p_next} sheets, actual wrapping at resonance = see table above')\n",
    "\n",
    "# ══════════════════════════════════════════════════════════════\n",
    "# CRITICAL: does x_q relate to the PRODUCT of wrapping fractions?\n",
    "# Or to the SUM? Or to some other combination?\n",
    "# ══════════════════════════════════════════════════════════════\n",
    "\n",
    "# At resonance, wrapping fractions are approximately:\n",
    "# f0=0, f1=0.5, f2=0.733, f3=0.857 = 0, 1/2, 11/15(?), 6/7\n",
    "# Actually: f1 = 105/210 = 1/2 exactly (p2=3 sheets, 1.5 of 3 wrap → 50%)\n",
    "# f2 = 154/210 = 11/15 = 0.7333 (p3=5 sheets, 3.67 of 5 wrap → 73.3%)\n",
    "# f3 = 180/210 = 6/7 exactly (p4=7 sheets, 6 of 7 wrap → 85.7%)\n",
    "\n",
    "f0, f1, f2, f3 = 0.0, 0.5, 154/210, 180/210\n",
    "print(f'\\nWrapping fractions at resonance: f0={f0}, f1={f1}, f2={f2:.4f}, f3={f3:.4f}')\n",
    "print(f'  f1 = 1/2 = 1/p1')\n",
    "print(f'  f2 = 11/15 (154/210)?  11/15={11/15:.4f} vs {f2:.4f}')\n",
    "print(f'  f3 = 6/7 = (p4-1)/p4')\n",
    "\n",
    "# Product: f1*f2*f3 = 0.5 × 0.733 × 0.857 = 0.314\n",
    "prod_f = f1 * f2 * f3\n",
    "print(f'\\n  Product f1*f2*f3 = {prod_f:.6f}')\n",
    "print(f'  Compare: 1/pi = {1/np.pi:.6f}')\n",
    "print(f'  Compare: phi(p3)/P4 = {4/P4:.6f}')\n",
    "\n",
    "# Sum of (1-f_k)*p_{k+1}: effective unwrapped sheets\n",
    "unwrap_sum = sum((1-f)*p for f, p in zip([f0,f1,f2,f3], primes))\n",
    "print(f'\\n  Sum of (1-f_k)*p_{{k+1}} = {unwrap_sum:.2f}')\n",
    "print(f'  Compare: omega(P4) = 4')\n",
    "\n",
    "# 4/7 = phi(p3)/p4. At the resonance:\n",
    "# - The R3 wrapping fraction is (p4-1)/p4 = 6/7 = phi(p4)/p4\n",
    "# - The exponent is phi(p3)/p4 = 4/7\n",
    "# The RATIO: exponent / wrapping_fraction = (4/7)/(6/7) = 4/6 = 2/3\n",
    "# = (p3-1)/(p4-1) = phi(p3)/phi(p4)\n",
    "print(f'\\n  Exponent/Wrapping = (4/7)/(6/7) = 4/6 = 2/3 = phi(p3)/phi(p4) = {4/6:.6f}')\n",
    "\n",
    "# Or: exponent = wrapping × phi(p3)/phi(p4)\n",
    "# x(R0) = f3 × phi(p3)/phi(p4) = (6/7) × (4/6) = 4/7 ✓\n",
    "print(f'  x(R0) = f_R3 × phi(p3)/phi(p4) = (6/7) × (4/6) = {(6/7)*(4/6):.6f} = 4/7 ✓')\n",
    "\n",
    "# This is a DERIVATION! x(R0) = f_R3 × phi(p3)/phi(p4)\n",
    "# where f_R3 = (p4-1)/p4 is the wrapping fraction at R3.\n",
    "#\n",
    "# In words: the mass exponent is the wrapping fraction at the outermost level\n",
    "# multiplied by the ratio of adjacent Euler totients.\n",
    "#\n",
    "# WHY? The wrapping fraction (p4-1)/p4 counts the fraction of branches\n",
    "# in the wrapping regime. The totient ratio phi(p3)/phi(p4) = 4/6\n",
    "# accounts for how the INNER level (p=5) modulates the outermost.\n",
    "# Together they give the effective exponent.\n",
    "\n",
    "print(f'\\n{\"=\"*70}')\n",
    "print(f'POTENTIAL DERIVATION:')\n",
    "print(f'  x(R0) = [(p4-1)/p4] × [phi(p3)/phi(p4)]')\n",
    "print(f'        = [(p4-1)/p4] × [(p3-1)/(p4-1)]')\n",
    "print(f'        = (p3-1)/p4')\n",
    "print(f'        = phi(p3)/p4')\n",
    "print(f'        = 4/7')\n",
    "print(f'  The (p4-1) in numerator and denominator CANCEL.')\n",
    "print(f'  So x(R0) = phi(p3)/p4 is just the algebraic identity.')\n",
    "print(f'  The wrapping fraction × totient ratio = simplified prime ratio.')\n",
    "print(f'  This is TAUTOLOGICAL — it doesn\\'t explain WHY, just rewrites.')\n",
    "print(f'{\"=\"*70}')\n",
    "\n",
    "# The REAL question remains: why does the cascade produce x(R0) = phi(p3)/p4\n",
    "# at the resonance kappa = 1/sqrt(P4)?\n",
    "# The kappa sweep PROVES it's not structural — it only works at this kappa.\n",
    "# So the answer must involve the relationship between:\n",
    "#   kappa = 1/sqrt(P4)\n",
    "#   ci_g2 = 11 (from CRT)\n",
    "#   wrapping boundary = ln(2)/kappa = 10.04\n",
    "#   and the covering map structure (p_{k+1} sheets at each level)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "concentric",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.12.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}


# NB160 — The Resonance

**Discovery (NB159)**: x_q(R0) = 4/7 ONLY at kappa = 1/sqrt(P4). The sheet
normalization is not a convention — it's the resonance condition that makes
mass exponents rational.

**Goal**: Understand the resonance mechanism. Why does kappa = 1/sqrt(P4)
produce rational exponents? What happens at the resonance point? And do
the scaling factors r_bs, r_tc also show resonance behavior?

**Key relationship**: kappa*P3 = P3/sqrt(P4) = 30/sqrt(210) = sqrt(30^2/210)
= sqrt(900/210) = sqrt(30/7). This is the spatial decay per generation spacing.

In [2]:
# S0 — The resonance structure
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa = 1.0 / np.sqrt(P4)
omega = 2 * np.pi

P = [1]
for p in primes:
    P.append(P[-1] * p)
P3 = P[3]  # 30

cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

# Generation crossings (quark Z2=0)
ci_g1, ci_g2, ci_g3 = 71, 11, 191

# ══════════════════════════════════════════════════════════════
# The resonance condition: what makes kappa = 1/sqrt(P4) special?
# ══════════════════════════════════════════════════════════════

print('=== The resonance point ===')
print(f'kappa = 1/sqrt({P4}) = {kappa:.10f}')
print(f'kappa^2 * P4 = {kappa**2 * P4:.10f}  (= 1 by sheet normalization)')

# Key spatial scales at the resonance:
print(f'\nSpatial scales at kappa = 1/sqrt(P4):')
print(f'  kappa * P3 = P3/sqrt(P4) = {kappa*P3:.6f} = sqrt(P3^2/P4) = sqrt({P3**2}/{P4}) = sqrt({P3**2/P4:.4f})')
print(f'  kappa * P4 = sqrt(P4) = {kappa*P4:.6f}')
print(f'  kappa * ci_g2 = {kappa*ci_g2:.6f} = {ci_g2}/sqrt({P4})')
print(f'  kappa * ci_g1 = {kappa*ci_g1:.6f} = {ci_g1}/sqrt({P4})')
print(f'  kappa * ci_g3 = {kappa*ci_g3:.6f} = {ci_g3}/sqrt({P4})')

# The wrapping boundary: where comb spacing = pi
# delta(ci) = 2*pi*e^{-kappa*ci} = pi when e^{-kappa*ci} = 1/2
# → ci_boundary = ln(2)/kappa = ln(2)*sqrt(P4)
ci_boundary = np.log(2) / kappa
print(f'\nWrapping boundary: ci = ln(2)/kappa = ln(2)*sqrt(P4) = {ci_boundary:.4f}')
print(f'  gen2 crossing: ci = {ci_g2}  (at {ci_g2/ci_boundary:.4f} of boundary)')
print(f'  gen2 is {ci_g2 - ci_boundary:.2f} past the wrapping boundary')
print(f'  → gen2 is RIGHT AT the transition. This is the resonance!')

# ══════════════════════════════════════════════════════════════
# The decay factors at generation crossings
# These are the FUNDAMENTAL quantities that determine everything
# ══════════════════════════════════════════════════════════════

print(f'\n=== Decay factors at generation crossings ===')
a_g1 = np.exp(-kappa * ci_g1)
a_g2 = np.exp(-kappa * ci_g2)
a_g3 = np.exp(-kappa * ci_g3)

print(f'  alpha(gen2, ci={ci_g2}) = e^(-{ci_g2}/sqrt({P4})) = {a_g2:.10f}')
print(f'  alpha(gen1, ci={ci_g1}) = e^(-{ci_g1}/sqrt({P4})) = {a_g1:.10f}')
print(f'  alpha(gen3, ci={ci_g3}) = e^(-{ci_g3}/sqrt({P4})) = {a_g3:.10f}')

# Ratios of decay factors:
print(f'\n  alpha(g2)/alpha(g3) = e^(kappa*180) = e^(180/sqrt(210)) = {a_g2/a_g3:.2f}')
print(f'  alpha(g1)/alpha(g2) = e^(-kappa*60) = e^(-60/sqrt(210)) = {a_g1/a_g2:.6f}')
print(f'  alpha(g1)/alpha(g3) = e^(-kappa*120) = e^(-120/sqrt(210)) = {a_g1/a_g3:.4f}')

# These spacing-dependent decay ratios are:
# e^{-kappa*n*P3} = e^{-n*P3/sqrt(P4)} = e^{-n*sqrt(P3^2/P4)}
# At the resonance: P3^2/P4 = 900/210 = 30/7
kP3 = kappa * P3  # = sqrt(30/7)
print(f'\n  kappa*P3 = sqrt(P3^2/P4) = sqrt({P3**2/P4:.6f}) = {kP3:.6f}')
print(f'  = sqrt(P3/p4) = sqrt({P3}/{p4}) = sqrt({P3/p4:.6f}) = {np.sqrt(P3/p4):.6f}')
print(f'  → The generation spacing decay = e^(-n*sqrt(P3/p4))')

# Specific values:
print(f'\n  e^(-1*sqrt(30/7)) = {np.exp(-np.sqrt(30/7)):.6f}  (gen3→gen2, 1 step)')  
print(f'  e^(-2*sqrt(30/7)) = {np.exp(-2*np.sqrt(30/7)):.6f}  (gen2→gen1, 2 steps)')
print(f'  e^(-3*sqrt(30/7)) = {np.exp(-3*np.sqrt(30/7)):.6f}  (gen3→gen1, 3 steps)')

# Wait: the spacings are 30 (g3→g2), 60 (g2→g1), 90 (g3→g1)
# In units of P3: 1, 2, 3 steps
# But ci_g2=11 and the first crossing is not at ci=0.
# The actual decay is e^{-kappa*ci}, not e^{-kappa*delta}.
# The CP ratio uses the ratio of decays, not the absolute values.

# ══════════════════════════════════════════════════════════════
# Fine-grained kappa sweep near the resonance
# ══════════════════════════════════════════════════════════════

from solenoid_jax import warmup
warmup()

cp_idx = {}
for name, (ch_a3, a7_g1, a7_g2) in CP_PAIRS.items():
    g1_mask = (a3 == ch_a3) & (a5 == 0) & (a7 == a7_g1)
    g2_mask = (a3 == ch_a3) & (a5 == 0) & (a7 == a7_g2)
    cp_idx[name] = (np.where(g1_mask)[0][0], np.where(g2_mask)[0][0])

def run_and_extract(kappa_val):
    sys0 = SolenoidSystem(primes=primes, kappa=kappa_val, epsilon=kappa_val)
    branches = sys0.all_branches()
    t_eval = cis.astype(float)
    res = sys0.integrate_all_branches(branches, t_eval, float(P4+1), backend='jax')
    
    rms = np.zeros((len(cis), 4))
    for idx in range(len(cis)):
        for k in range(4):
            Rk = np.array([res[br][idx, k] for br in branches])
            Rk_w = np.mod(Rk, 2*np.pi)
            Rk_w[Rk_w > np.pi] -= 2*np.pi
            rms[idx, k] = np.sqrt(np.mean(Rk_w**2))
    
    # CP ratios at all levels for quark pair
    i1, i2 = cp_idx['QUARK']
    cp_q = rms[i1] / rms[i2]
    
    # Also get all 6 quark gen crossings (for r_bs, r_tc)
    gen_map = {0:1, 3:1, 1:2, 4:2, 2:3, 5:3}
    gen_rms = {}
    for a7v in [0, 4, 2]:  # gen1 Z2=0, gen2 Z2=0, gen3 Z2=0
        idx = np.where((a3 == 1) & (a5 == 0) & (a7 == a7v))[0][0]
        gen_rms[gen_map[a7v]] = rms[idx]
    
    return cp_q, gen_rms, rms

# Fine sweep: 0.9 to 1.1 in 0.01 steps
kf_fine = np.arange(0.90, 1.105, 0.01)
print(f'\n=== Fine kappa sweep: {len(kf_fine)} values from 0.90 to 1.10 ===')

sweep_data = []
for kf in kf_fine:
    kv = kappa * kf
    cp_q, gen_rms, rms_full = run_and_extract(kv)
    
    # x_q at each level
    x_q_R0 = np.log(20.0) / np.log(cp_q[0]) if cp_q[0] > 1.01 else np.nan
    x_q_R3 = np.log(20.0) / np.log(cp_q[3]) if cp_q[3] > 1.01 else np.nan
    
    # For r_bs and r_tc: compute WHAT mass ratios the cascade predicts
    # at each generation crossing. If we use x_q(R3) for all mass ratios:
    # m_s/m_d = CP_R3^{x_q} = 20 (by definition)
    # But what about other generation comparisons?
    # The gen1/gen3 and gen2/gen1 ratios at R3 give additional mass info.
    r_g1 = gen_rms[1]  # gen1 Z2=0 (ci=71)
    r_g2 = gen_rms[2]  # gen2 Z2=0 (ci=11)
    r_g3 = gen_rms[3]  # gen3 Z2=0 (ci=191)
    
    # Cross-gen ratios at R3
    cp23_R3 = r_g2[3] / r_g3[3]  # gen2/gen3 at R3
    cp13_R3 = r_g1[3] / r_g3[3]  # gen1/gen3 at R3
    cp21_R3 = r_g2[3] / r_g1[3]  # gen2/gen1 at R3
    
    # ln ratios / ln(CP pair)
    if cp23_R3 > 1.01:
        ln_ratio_13 = np.log(cp13_R3) / np.log(cp23_R3)
        ln_ratio_21 = np.log(cp21_R3) / np.log(cp23_R3)
    else:
        ln_ratio_13 = np.nan
        ln_ratio_21 = np.nan
    
    sweep_data.append({
        'kf': kf, 'kappa': kv,
        'x_q_R0': x_q_R0, 'x_q_R3': x_q_R3,
        'cp23_R3': cp23_R3, 'cp13_R3': cp13_R3, 'cp21_R3': cp21_R3,
        'ln_13': ln_ratio_13, 'ln_21': ln_ratio_21,
    })

print(f'\n{"kf":>5} {"x_q(R0)":>10} {"dev 4/7":>10} {"CP_R3":>8} {"ln13/ln23":>10} {"ln21/ln23":>10}')
for d in sweep_data:
    dev = abs(d['x_q_R0'] - 4/7) / (4/7) * 100 if not np.isnan(d['x_q_R0']) else 999
    print(f'{d["kf"]:5.2f} {d["x_q_R0"]:10.6f} {dev:9.3f}% {d["cp23_R3"]:8.4f} {d["ln_13"]:10.6f} {d["ln_21"]:10.6f}')

# ══════════════════════════════════════════════════════════════
# KEY TEST: At the resonance (kf=1.0), what are the cross-gen
# log-ratios? These determine r_bs and r_tc.
# Do they show any special behavior at the resonance?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Cross-gen structure at and near resonance ===')
print(f'  ln(gen1/gen3) / ln(gen2/gen3) at R3:')
print(f'  This ratio is level-independent for the CP pair.')
print(f'  But it changes with kappa for the cross-gen ratios.')
print(f'  At the resonance (kf=1.0): r_bs = 1.269, so we need')
print(f'  some combination of cross-gen ratios to give 1.269 and 1.639.')
print(f'')

# The cross-gen log-ratio ln(gen1/gen3)/ln(gen2/gen3) at R3 is about 0.39
# This is NOT r_bs. But it's a spatial property that changes with kappa.
# Maybe r_bs is a FUNCTION of this spatial property?

# Let's check: does the cross-gen structure show a resonance at kf=1.0?
print(f'\nCross-gen log-ratio ln(gen1/gen3)/ln(gen2/gen3) vs kappa:')
for d in sweep_data:
    if not np.isnan(d['ln_13']):
        # At kf=1.0, ln_13 ≈ 0.392. Does this vary smoothly?
        print(f'  kf={d["kf"]:.2f}: ln13={d["ln_13"]:.6f}  ln21={d["ln_21"]:.6f}  sum={d["ln_13"]+d["ln_21"]:.6f}')

# Note: ln(gen1/gen3)/ln(gen2/gen3) + ln(gen2/gen1)/ln(gen2/gen3) = 1 always
# because ln(gen1/gen3) = ln(gen2/gen3) - ln(gen2/gen1)
# Actually: ln(gen1/gen3) + ln(gen2/gen1) = ln(gen1/gen3 * gen2/gen1) = ln(gen2/gen3)
# So ln13/ln23 + ln21/ln23 = 1. The two ratios are complementary.

print(f'\nNote: ln13/ln23 + ln21/ln23 = 1 always (complementary)')
print(f'So there is only ONE independent cross-gen quantity: ln13/ln23.')
print(f'At kf=1.0: ln13/ln23 = {[d for d in sweep_data if abs(d["kf"]-1.0)<0.001][0]["ln_13"]:.6f}')

=== The resonance point ===
kappa = 1/sqrt(210) = 0.0690065559
kappa^2 * P4 = 1.0000000000  (= 1 by sheet normalization)

Spatial scales at kappa = 1/sqrt(P4):
  kappa * P3 = P3/sqrt(P4) = 2.070197 = sqrt(P3^2/P4) = sqrt(900/210) = sqrt(4.2857)
  kappa * P4 = sqrt(P4) = 14.491377
  kappa * ci_g2 = 0.759072 = 11/sqrt(210)
  kappa * ci_g1 = 4.899465 = 71/sqrt(210)
  kappa * ci_g3 = 13.180252 = 191/sqrt(210)

Wrapping boundary: ci = ln(2)/kappa = ln(2)*sqrt(P4) = 10.0447
  gen2 crossing: ci = 11  (at 1.0951 of boundary)
  gen2 is 0.96 past the wrapping boundary
  → gen2 is RIGHT AT the transition. This is the resonance!

=== Decay factors at generation crossings ===
  alpha(gen2, ci=11) = e^(-11/sqrt(210)) = 0.4681005689
  alpha(gen1, ci=71) = e^(-71/sqrt(210)) = 0.0074505645
  alpha(gen3, ci=191) = e^(-191/sqrt(210)) = 0.0000018875

  alpha(g2)/alpha(g3) = e^(kappa*180) = e^(180/sqrt(210)) = 247999.02
  alpha(g1)/alpha(g2) = e^(-kappa*60) = e^(-60/sqrt(210)) = 0.015917
  alpha(g1)/alpha(